# PokeScanner training on Colab

Runs the trainer from the repo on a free T4. Roughly 30 to 60 minutes
for 30 epochs of ConvNeXt-Tiny over ~7,000 images.

**Runtime > Change runtime type > T4 GPU** before you start, or it will
train on CPU and take all day.

Order: setup, get the data, train, look at the result, download weights.


## 1. Setup


In [ ]:
!nvidia-smi -L || echo 'NO GPU - set Runtime > Change runtime type > T4 GPU'


In [ ]:
# The repo brings the trainer, the augmentation code and the label map.
!git clone --depth 1 https://github.com/Phennnn/Pokescanner.git
%cd Pokescanner
!pip install -q timm
import torch, timm
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())


## 2. Get the images

Pick **one** of the next two cells.

**Option A (recommended)** builds the dataset here from the PokeAPI
sprite repo. Nothing to upload, and it gives about nine views per
species including official artwork and HOME renders, which look far
more like real photos than a 96x96 game sprite does.

**Option B** uploads a zip you built locally, for when you already have
a bigger set on your machine.


In [ ]:
# Option A: build the dataset here (about 10 minutes)
!python tools/fetch_sprites.py


In [ ]:
# Option B: upload a zip made with `python tools/make_colab_bundle.py`
# from google.colab import files
# up = files.upload()
# name = next(iter(up))
#
# # Zips written on Windows can carry \-separated paths, which
# # unzip treats as part of the filename rather than as folders. That
# # is what produced one flat directory of 'data\images\x.png'
# # files last time. Rebuilding each path here fixes it either way.
# import zipfile, pathlib
# with zipfile.ZipFile(name) as z:
#     for info in z.infolist():
#         if info.is_dir():
#             continue
#         fixed = pathlib.Path(info.filename.replace('\\', '/'))
#         target = pathlib.Path('.') / fixed
#         target.parent.mkdir(parents=True, exist_ok=True)
#         target.write_bytes(z.read(info))
# print('extracted')


In [ ]:
# What we ended up with
from pathlib import Path
dirs = [d for d in Path('data/images').iterdir() if d.is_dir()]
counts = sorted(len(list(d.glob('*'))) for d in dirs)
print(f'{len(dirs)} classes, {sum(counts)} images')
print(f'per class: min {counts[0]}, median {counts[len(counts)//2]}, max {counts[-1]}')
print(f'{sum(1 for c in counts if c < 2)} classes with <2 images (no validation split)')


## 3. Train

`--background-prob 0.5` composites half the training sprites into
cluttered synthetic scenes. That is the training-side half of the
domain-gap fix; the inference-side half already ships in
`pokescanner/vision.py`.

The split is stratified per class, so the validation accuracy printed
here is measured on a balanced held-out set rather than the accidental
subset the old global shuffle produced.


In [ ]:
!python model/train.py \
    --arch convnext_tiny \
    --epochs 30 \
    --batch-size 32 \
    --background-prob 0.5 \
    --workers 2


### Worth running afterwards

```
!python model/train.py --arch efficientnet_b2 --epochs 30
!python model/train.py --arch convnext_tiny --background-prob 0
!python model/train.py --arch convnext_tiny --epochs 60 --resume
```

The middle one ablates the background augmentation. It is the run that
tells you how much of any gain came from the augmentation rather than
from switching backbone, which is the question a reviewer would ask.


## 4. Check what you got

Top-1 alone hides the useful detail. This prints confusion pairs, the
classes that are never right, and a calibration table showing whether
the confidence number means anything.


In [ ]:
!python model/evaluate.py --weights model/weights/best_model_convnext_tiny.pth


In [ ]:
# The harder question: cluttered scenes rather than clean sprites
!python model/evaluate.py --weights model/weights/best_model_convnext_tiny.pth \
    --scenes --limit 300


In [ ]:
# And the inference-pipeline comparison on the new weights
!python tools/benchmark.py --n 300 --views 4 \
    --weights model/weights/best_model_convnext_tiny.pth


## 5. Bring the weights home


In [ ]:
from google.colab import files
files.download('model/weights/best_model_convnext_tiny.pth')


Drop that file into `model/weights/` locally. The apps pick it up on
their own, because the checkpoint records its own architecture and input
size. To switch back for a comparison:

```bash
POKESCANNER_WEIGHTS=best_model_b2.pth python app/pokedex.py
```


---
### If it goes wrong

| symptom | cause |
|---|---|
| `CUDA out of memory` | lower `--batch-size` to 16 |
| every class has 1 image | the fetch cell did not run, or ran elsewhere |
| `val top1 0.00%`, file named `last_model_*` | no validation set existed, so nothing could be selected on |
| session disconnected mid-run | rerun the train cell with `--resume` |
| one flat folder of `data\images\...` files | the Windows zip issue, use Option A |
